## GPT n50 analysis

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import csv

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [3]:
DATA_FILE = "../../results/n50_examples_large_v01/gpt_v02/gpt_b10_run01.csv"

## Vastustega df

In [24]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")
df1["verb_compound"] = df1["verb_compound"].fillna("")

In [5]:
len(df1[df1["classification"]=="yes"]) # koht

6336

In [6]:
len(df1[df1["classification"]=="no"]) # mitte koht

3664

In [7]:
df1["classification2"].unique()

array(['loc', 'actor', 'time', 'event', 'UNK'], dtype=object)

In [8]:
df1.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)

,verb,verb_compound,morph_case,count
113,jääma,NaN,adit,42
323,nägema,välja,in,41
89,jooksma,NaN,el,40
544,unistama,NaN,el,39
112,jääma,NaN,abl,38
...,...,...,...,...
570,viima,läbi,el,1
157,keelustama,NaN,el,1
464,sõitma,"tagant, otsa",ad,1
480,tagurdama,otsa,ad,1


### Tabel: lõksude jaoks iga labeli arv ja protsent näidetest

In [25]:
a1 = df1.groupby(["verb", "verb_compound", 'morph_case', "classification2"], dropna=False, as_index=False).size().reset_index().sort_values('verb', ascending=False)
a1["verb_compound"] = a1["verb_compound"].fillna("")

a1["ex_percentage"] = a1.groupby(["verb", "verb_compound", 'morph_case'])["size"].transform(lambda x: x/x.sum()*100)
a1 = a1.sort_values(["verb", "verb_compound", 'morph_case'])

# aggregate
summary = (
    a1.groupby(["verb", "verb_compound", "morph_case"], as_index=False)
      .agg(
          ex_count=("size", "sum"),

          # absolute counts
          loc_count=("size", lambda x: x[a1.loc[x.index, "classification2"] == "loc"].sum()),
          actor_count=("size", lambda x: x[a1.loc[x.index, "classification2"] == "actor"].sum()),
          time_count=("size", lambda x: x[a1.loc[x.index, "classification2"] == "time"].sum()),
          event_count=("size", lambda x: x[a1.loc[x.index, "classification2"] == "event"].sum()),
          unk_count=("size", lambda x: x[a1.loc[x.index, "classification2"] == "UNK"].sum()),
          
          # labels info
          label_count=("classification2", lambda x: x.nunique()),
          labels=("classification2", lambda x: tuple(sorted(x.unique())))
          
      ).pipe(lambda df: df[df["label_count"]>=1])
)

# --- compute percentages safely ---
summary["loc_percent"] = (
    summary["loc_count"] / summary["ex_count"] * 100
)

summary["actor_percent"] = (
    summary["actor_count"] / summary["ex_count"] * 100
)

summary["time_percent"] = (
    summary["time_count"] / summary["ex_count"] * 100
)

summary["event_percent"] = (
    summary["event_count"] / summary["ex_count"] * 100
)

summary["unk_percent"] = (
    summary["unk_count"] / summary["ex_count"] * 100
)

In [26]:
summary

,verb,verb_compound,morph_case,ex_count,loc_count,actor_count,time_count,event_count,unk_count,label_count,labels,loc_percent,actor_percent,time_percent,event_percent,unk_percent
0,abielluma,,in,14,9,0,4,0,1,3,"(UNK, loc, time)",64.285714,0.000000,28.571429,0.000000,7.142857
1,aeguma,,in,3,0,0,0,0,3,1,"(UNK,)",0.000000,0.000000,0.000000,0.000000,100.000000
2,aitama,,in,19,17,0,0,1,1,3,"(UNK, event, loc)",89.473684,0.000000,0.000000,5.263158,5.263158
3,ajama,,all,35,21,7,0,0,7,3,"(UNK, actor, loc)",60.000000,20.000000,0.000000,0.000000,20.000000
4,ajama,,el,26,10,4,0,0,12,3,"(UNK, actor, loc)",38.461538,15.384615,0.000000,0.000000,46.153846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604,ütlema,,adit,8,7,1,0,0,0,2,"(actor, loc)",87.500000,12.500000,0.000000,0.000000,0.000000
605,ütlema,,all,29,12,15,0,0,2,3,"(UNK, actor, loc)",41.379310,51.724138,0.000000,0.000000,6.896552
606,ütlema,,el,24,19,2,0,0,3,3,"(UNK, actor, loc)",79.166667,8.333333,0.000000,0.000000,12.500000
607,ütlema,välja,in,7,6,0,0,0,1,2,"(UNK, loc)",85.714286,0.000000,0.000000,0.000000,14.285714


In [27]:
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
SUMMARY_FILE = SUMMARY_DIR+ "n50_verbcase_label_distribution.csv"
summary.to_csv(SUMMARY_FILE, encoding="utf-8", index=False, sep=",")

In [117]:
def get_examples(filtered, label1, label2):

    pairs = filtered[['verb', 'verb_compound', 'morph_case']].drop_duplicates()

    results = []
    final_df = pd.DataFrame()
    
    if len(pairs) != 0:
        for _, row in pairs.iterrows():
            verb = row['verb']
            comp = row['verb_compound']
            case = row['morph_case']

            subset = df1[
                (df1['verb'] == verb) &
                (df1['verb_compound'] == comp) &
                (df1['morph_case'] == case)
            ]

            l1 = subset[subset['classification2'] == label1].sample(
                n=min(5, len(subset[subset['classification2']==label1])),
                random_state=1
            )

            l2 = subset[subset['classification2'] == label2].sample(
                n=min(5, len(subset[subset['classification2']==label2])),
                random_state=1
            )

            results.append(l1)
            results.append(l2)

        final_df = pd.concat(results, ignore_index=True)

        cols = ['verb', 'verb_compound', 'morph_case', 'classification2'] + \
               [c for c in final_df.columns if c not in ['verb','verb_compound','morph_case','classification2']]

        # Reorder the dataframe
        final_df = final_df[cols]


        final_df['sort_class'] = final_df['classification2'].apply(lambda x: 0 if x == label1 else 1)

        # Sort by verb, verb_compound, then helper column, then classification2
        final_df = final_df.sort_values(
            by=['verb', 'verb_compound', 'sort_class', 'classification2']
        )

        # Drop helper column if you want
        final_df = final_df.drop(columns=['sort_class'])


        # sort the final table
        #final_df = final_df.sort_values(
        #    by=['verb', 'verb_compound', 'classification2']
        #)

    return final_df

In [49]:
loc_vs_df = pd.DataFrame()

## loc ja actor

In [50]:
min_total = 10
min_raw = 5
min_share = 30

filtered2 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["loc_count"] >= min_raw) &
    (summary["actor_count"] >= min_raw) &
    (summary["loc_percent"] >= min_share) &
    (summary["actor_percent"] >= min_share)
]

In [51]:
loc_vs_df = pd.concat([loc_vs_df, get_examples(filtered2, "loc", "actor")], ignore_index=True) 

## loc ja time

In [53]:
min_total = 10
min_raw = 2
min_share = 30

filtered4 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["loc_count"] >= min_raw) &
    (summary["time_count"] >= min_raw) &
    (summary["loc_percent"] >= min_share) &
    (summary["time_percent"] >= min_share)
]

In [54]:
loc_vs_df = pd.concat([loc_vs_df, get_examples(filtered4, "loc", "time")], ignore_index=True) 

## loc ja event

In [55]:
min_total = 10
min_raw = 5
min_share = 30

filtered5 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["loc_count"] >= min_raw) &
    (summary["event_count"] >= min_raw) &
    (summary["loc_percent"] >= min_share) &
    (summary["event_percent"] >= min_share)
]

In [56]:
loc_vs_df = pd.concat([loc_vs_df, get_examples(filtered5, "loc", "event")], ignore_index=True) 

## loc ja UNK

In [57]:
min_total = 10
min_raw = 5
min_share = 30

filtered6 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["loc_count"] >= min_raw) &
    (summary["unk_count"] >= min_raw) &
    (summary["loc_percent"] >= min_share) &
    (summary["unk_percent"] >= min_share)
]

In [58]:
loc_vs_df = pd.concat([loc_vs_df, get_examples(filtered6, "loc", "UNK")], ignore_index=True) 

In [59]:
loc_vs_df

,verb,verb_compound,morph_case,classification2,sentence_id,head_id,head_loc,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract
0,astuma,,el,loc,10991082,17603700,19,jalg,jalust,Vene kultuurikeskuses peetud « Miss Teini » võistlusel astus lavale kümne neiu asemel seitse külmetushaigus niitis kolm tütarlast jalust .,NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN
1,astuma,,el,loc,6796324,10934290,4,esiuks,esiuksest,""" Astusin trolli esiuksest , täksisin pileti ja kuna troll oli rahvast täis , jäin esimese posti juurde seisma .",NaN,NaN,NaN,yes,"The phrase 'esiuksest' was classified as 'yes' because it specifies a location, indicating where the speaker entered the troll.",NaN,NaN,NaN,NaN,NaN
2,astuma,,el,loc,4316912,6944541,7,saatkond,saatkonnast,"Samuti astus ta läbi Vene föderatsiooni saatkonnast , kus nii talle kui ka naisele ja lapsele väljastati kolme tunniga viisad Peterburi külastamiseks .",NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN
3,astuma,,el,loc,11183528,17913978,4,repressioon,repressioonidest,"Peremehed , kes repressioonidest hoolimata ikkagi ei astunud kolhoosi 1949. aasta jooksul , maksustati võrreldes eelmise aastaga kolmekordse põllumajandusmaksuga .",NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN
4,astuma,,el,loc,3279008,5270566,7,Türgi,Türgist,Kuni kangi juurde astub Nurcan Taylan Türgist .,NaN,location,LOC,yes,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,õnnestuma,,el,UNK,2570839,4125080,10,kaotusseis,kaotusseisust,"Poolfinaalis sai eestlane töövõidu valgevenelase Igor Makarovi üle , kaotusseisust õnnestus pääseda pool minutit enne lõpugongi .",NaN,state,NaN,no,"The phrase 'kaotusseisust' refers to a state or situation and not an actual location, so it is not an adverbial of place.",no,no,no,no,no
636,õnnestuma,,el,UNK,2535832,4068457,7,kulu,kuludest,Näiteks köögi või vannitoa põhjaliku uuendamise kuludest õnnestus 2001. aastal ajakirja hinnangul tagasi saada u 80 protsenti ja kodukontori lisamisest vaid 50 protsendi ringis .,NaN,NaN,NaN,no,"The phrase 'kuludest' refers to costs, which do not indicate a location, so it is not an adverbial of place.",no,no,no,no,no
637,õnnestuma,,el,UNK,6138231,9852871,9,kompromiss,kompromissist,Sakslaste ühistööna valminud taskuteatmik on suurepärane näide õnnestunud kompromissist .,NaN,NaN,NaN,no,"The phrase 'kompromissist' refers to the source or origin (compromise) and not a location, so it is not adverbial of place.",no,no,no,no,no
638,õnnestuma,,el,UNK,471296,739368,14,jõukus,jõukusest,Maa ja linna vahe kadumine säärasel hõredalt asustatud maal nagu Soome pole suhtelisest jõukusest hoolimata ilmselt õnnestunud .,NaN,NaN,NaN,no,"The phrase 'jõukusest' refers to a cause or reason (despite relative affluence), not a location, so it is not an adverbial of place.",no,no,no,no,no


In [61]:
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
LOC_FILE = SUMMARY_DIR+ "n50_verbcase_label_loc_vs_other.csv"
loc_vs_df.to_csv(LOC_FILE, encoding="utf-8", index=False, sep=",")

In [63]:
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
LOC_FILE2 = SUMMARY_DIR+ "n50_verbcase_label_loc_vs_other_readable_sample.csv"
loc_vs_df[:100].to_csv(LOC_FILE2, encoding="utf-8", index=False, sep=",")

In [72]:
summary2 = summary[summary["labels"].map(lambda x: "UNK" in x) & (summary["unk_percent"]<60)].copy()
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
SUMMARY_FILE2 = SUMMARY_DIR+ "n50_verbcase_label_dist_unk_below60.csv"
summary2.to_csv(SUMMARY_FILE2, encoding="utf-8", index=False, sep=",")

In [118]:
unk_vs_df = pd.DataFrame()

## UNK ja actor

In [119]:
min_total = 10
min_raw = 2
min_share = 30

filtered11 = summary2[
    (summary2["ex_count"] >= min_total) &
    (summary2["unk_count"] >= min_raw) &
    (summary2["actor_count"] >= min_raw) &
    (summary2["unk_percent"] >= min_share) &
    (summary2["actor_percent"] >= min_share)
]

In [120]:
unk_vs_df = pd.concat([unk_vs_df, get_examples(filtered11, "UNK", "actor")], ignore_index=True) 

## UNK ja time

In [121]:
min_total = 10
min_raw = 2
min_share = 30

filtered12 = summary2[
    (summary2["ex_count"] >= min_total) &
    (summary2["unk_count"] >= min_raw) &
    (summary2["time_count"] >= min_raw) &
    (summary2["unk_percent"] >= min_share) &
    (summary2["time_percent"] >= min_share)
]

In [122]:
unk_vs_df = pd.concat([unk_vs_df, get_examples(filtered12, "UNK", "time")], ignore_index=True) 

## UNK ja event

In [123]:
min_total = 10
min_raw = 2
min_share = 30

filtered13 = summary2[
    (summary2["ex_count"] >= min_total) &
    (summary2["unk_count"] >= min_raw) &
    (summary2["event_count"] >= min_raw) &
    (summary2["unk_percent"] >= min_share) &
    (summary2["event_percent"] >= min_share)
]

In [124]:
unk_vs_df = pd.concat([unk_vs_df, get_examples(filtered13, "UNK", "event")], ignore_index=True) 

## UNK ja loc

In [125]:
min_total = 10
min_raw = 2
min_share = 30

filtered14 = summary2[
    (summary2["ex_count"] >= min_total) &
    (summary2["unk_count"] >= min_raw) &
    (summary2["loc_count"] >= min_raw) &
    (summary2["unk_percent"] >= min_share) &
    (summary2["loc_percent"] >= min_share)
]

In [126]:
unk_vs_df = pd.concat([unk_vs_df, get_examples(filtered14, "UNK", "loc")], ignore_index=True) 

In [127]:
unk_vs_df

,verb,verb_compound,morph_case,classification2,sentence_id,head_id,head_loc,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract
0,kommenteerima,,el,UNK,15349472,23974102,18,otsus,otsusest,""" Arvan , et Murutar oleks saate juhtimisega ka üksi suurepäraselt toime tulnud , "" kommenteeris Murutari otsusest teha saadet koos Tammeriga Pihlamägi .",NaN,NaN,NaN,no,"The phrase 'otsusest' refers to the decision, which is an abstract concept and not a location, so it is not classified as an adverbial of place.",no,no,no,no,no
1,kommenteerima,,el,UNK,1278042,2031361,8,saavutamine,saavutamisest,Palestiinlased ei ole veel Iisraeli teadet kokkuleppe saavutamisest kommenteerinud .,NaN,NaN,NaN,no,"The phrase 'saavutamisest' refers to an action or process, not specifying any place, so it is classified as 'no'.",no,no,no,no,no
2,kommenteerima,,el,UNK,4168369,6708738,14,säilitamine,säilitamisest,Mihhail Petrov kommenteeris Euroopa vene kogukondade nõukogu poolt Riias esitletud Vene doktriini rahvuse säilitamisest .,NaN,NaN,NaN,no,"The phrase 'säilitamisest' was not classified as an adverbial of place because it does not describe a location or spatial relation; instead, it relates to the concept of preservation.",no,no,no,no,no
3,kommenteerima,,el,actor,5360794,8598798,1,kohalolija,Kohalolijatest,"Kohalolijatest kommenteeris Office'i eestikeelsust Megale tehnokratt Peeter Marvet , kes süüdistas riiki ühtse eestikeelse arvutiterminoloogia välja töötamata jätmises .",NaN,alive,NaN,no,"The phrase 'Kohalolijatest' refers to attendees or participants present at a location, but it does not describe a specific place, so it is not classified as an adverbial of place.",no,yes,no,no,no
4,kommenteerima,,el,actor,11398301,18265871,18,kaasmaalane,kaasmaalasest,""" Kui teaksin , millega ta üllatab , siis see poleks enam üllatus , "" kommenteeris Goes kaasmaalasest kolleegi sõnu .",NaN,alive,NaN,no,"The word 'kaasmaalasest' indicates a relationship or origin and does not refer to a physical place, so it is not an adverbial of place.",no,yes,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561,õnnestuma,,el,loc,12176377,19495224,6,põleng,põlengust,Ootamatult ja teadmata põhjusel alanud põlengust õnnestus majas elanud Arturil päästa oma õde Natalja .,NaN,event,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN
562,õnnestuma,,el,loc,10688543,17127995,2,Rõuge,Rõugest,"Võrumaalt Rõugest õnnestus teada saada , et mõniteist aastat tagasi elas sealmail Eha Kulli nimeline naine , kelle vend elas samuti Rõuges .",NaN,location,LOC,yes,NaN,NaN,NaN,NaN,NaN,NaN
563,õnnestuma,,el,loc,771520,1230365,10,ravivõimalus,ravivõimalustest,"Ent on raske uskuda , et meil õnnestuks praegustest ravivõimalustest midagi enamat välja visata kui hollandlastel , kes ei jätnud kõrvale suurt enamat peale katseklaasis viljastamise .",NaN,NaN,NaN,yes,"The phrase 'ravivõimalustest' involves treatment possibilities and is used in the sense of extraction from a conceptual 'place,' hence it is classified as an adverbial of place.",NaN,NaN,NaN,NaN,NaN
564,õnnestuma,,el,loc,9386506,15072731,2,tagauks,tagauksest,Vaid tagauksest ja kellegi teise abil õnnestub ratastoolis ukerdada polikliinikusse .,NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
UNK_FILE = SUMMARY_DIR+ "n50_verbcase_label_unk_vs_other.csv"
unk_vs_df.to_csv(UNK_FILE, encoding="utf-8", index=False, sep=",")

In [130]:
SUMMARY_DIR = "../../results/n50_examples_large_v01/"
UNK_FILE2 = SUMMARY_DIR+ "n50_verbcase_label_unk_vs_other_readable_sample.csv"
unk_vs_df[:100].to_csv(UNK_FILE2, encoding="utf-8", index=False, sep=",")

## actor ja time

In [25]:
min_total = 10
min_raw = 2
min_share = 30

filtered3 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["time_count"] >= min_raw) &
    (summary["actor_count"] >= min_raw) &
    (summary["time_percent"] >= min_share) &
    (summary["actor_percent"] >= min_share)
]

In [26]:
filtered3

,verb,verb_compound,morph_case,ex_count,loc_count,actor_count,time_count,event_count,unk_count,label_count,labels,loc_percent,actor_percent,time_percent,event_percent,unk_percent


## actor ja event

In [27]:
min_total = 10
min_raw = 2
min_share = 30

filtered6 = summary[
    (summary["ex_count"] >= min_total) &
    (summary["actor_count"] >= min_raw) &
    (summary["event_count"] >= min_raw) &
    (summary["actor_percent"] >= min_share) &
    (summary["event_percent"] >= min_share)
]

In [28]:
filtered6

,verb,verb_compound,morph_case,ex_count,loc_count,actor_count,time_count,event_count,unk_count,label_count,labels,loc_percent,actor_percent,time_percent,event_percent,unk_percent
